# Pensjonsdemografi og pensjonsvolum

Denne notebooken analyserer ferdige Gold-data fra Pensjon Lakehouse-pipelinen.

Rapporten viser:

1. Utvikling i gjennomsnittlig pensjonsandel over tid
2. Kommuner med høyest andel innbyggere 55+
3. Næringer med høyest estimert pensjonsvolum
4. Sammenheng mellom antall lønnstakere og månedslønn per næring

Forutsetning:

```bash
python main.py
```

må være kjørt først, slik at Gold-data finnes under:

```text
/tmp/pensjon_lakehouse/gold/
```


## 1. Imports og paths

Notebooken leser kun ferdige Parquet-filer fra Gold-laget. Den henter ikke data fra SSB og kjører ikke selve lakehouse-pipelinen på nytt.


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt


LAKE = Path("/tmp/pensjon_lakehouse")
GOLD = LAKE / "gold"
REPORTS = LAKE / "reports"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

GOLD


## 2. Sjekk at Gold-data finnes

Hvis denne cellen feiler, kjør `python main.py` fra prosjektroten først.


In [ ]:
required_files = [
    GOLD / "pensjonsandel_trend.parquet",
    GOLD / "top_kommuner_pensjonsalder.parquet",
    GOLD / "naering_pensjonsvolum.parquet",
]

missing_files = [file for file in required_files if not file.exists()]

if missing_files:
    raise FileNotFoundError(
        "Mangler Gold-filer. Kjør `python main.py` først.\n"
        + "\n".join(str(file) for file in missing_files)
    )

for file in required_files:
    print(file)


## 3. Koble til DuckDB

Vi bruker DuckDB direkte i notebooken for å lese Parquet-filene.


In [ ]:
db = duckdb.connect()


## 4. Pensjonsandel over tid

Denne delen viser hvordan gjennomsnittlig andel innbyggere 55+ utvikler seg over tid.


In [ ]:
df_trend = db.execute(f"""
    SELECT
        year,
        snitt_pensjonsandel_pst,
        total_55_pluss,
        total_befolkning
    FROM read_parquet('{GOLD}/pensjonsandel_trend.parquet')
    ORDER BY year
""").fetchdf()

df_trend


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    df_trend["year"],
    df_trend["snitt_pensjonsandel_pst"],
    marker="o",
)

plt.title("Gjennomsnittlig pensjonsandel over tid")
plt.xlabel("År")
plt.ylabel("Pensjonsandel 55+ (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Kommuner med høyest andel 55+

Her ser vi hvilke kommuner som har størst andel innbyggere i aldersgruppen 55+.


In [ ]:
df_kommuner = db.execute(f"""
    WITH top10 AS (
        SELECT
            kommune_label,
            total_befolkning,
            pension_age_befolkning,
            CAST(ROUND(pension_age_share * 100, 1) AS DECIMAL(5, 1)) AS andel_55_pluss
        FROM read_parquet('{GOLD}/top_kommuner_pensjonsalder.parquet')
        ORDER BY pension_age_share DESC
        LIMIT 10
    )
    SELECT *
    FROM top10
    ORDER BY andel_55_pluss ASC
""").fetchdf()

df_kommuner


In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    df_kommuner["kommune_label"],
    df_kommuner["andel_55_pluss"],
)

plt.title("Kommuner med høyest andel innbyggere 55+")
plt.xlabel("Andel 55+ (%)")
plt.ylabel("Kommune")
plt.tight_layout()
plt.show()


## 6. Næringer med høyest estimert pensjonsvolum

Estimert pensjonsvolum er beregnet i pipelinen som:

```text
lønnstakere × månedslønn × 12 × 0.02
```

`Alle næringer` ekskluderes her fordi det er en totalsum, ikke en enkelt næring.


In [ ]:
df_naering = db.execute(f"""
    WITH top10 AS (
        SELECT
            naering_label,
            lonsstakere,
            manedslonn,
            estimert_pensjonsvolum,
            estimert_pensjonsvolum / 1000000000 AS estimert_pensjonsvolum_mrd
        FROM read_parquet('{GOLD}/naering_pensjonsvolum.parquet')
        WHERE naering_label <> 'Alle næringer'
        ORDER BY estimert_pensjonsvolum DESC
        LIMIT 10
    )
    SELECT *
    FROM top10
    ORDER BY estimert_pensjonsvolum_mrd ASC
""").fetchdf()

df_naering


In [ ]:
plt.figure(figsize=(11, 6))

plt.barh(
    df_naering["naering_label"],
    df_naering["estimert_pensjonsvolum_mrd"],
)

plt.title("Næringer med høyest estimert pensjonsvolum")
plt.xlabel("Estimert pensjonsvolum, mrd. kr")
plt.ylabel("Næring")
plt.tight_layout()
plt.show()


## 7. Lønnstakere og månedslønn per næring

Denne figuren gir et ekstra blikk på hva som driver pensjonsvolumet: mange ansatte, høy månedslønn, eller en kombinasjon.


In [ ]:
df_naering_scatter = db.execute(f"""
    SELECT
        naering_label,
        lonsstakere,
        manedslonn,
        estimert_pensjonsvolum / 1000000000 AS estimert_pensjonsvolum_mrd
    FROM read_parquet('{GOLD}/naering_pensjonsvolum.parquet')
    WHERE naering_label <> 'Alle næringer'
    ORDER BY estimert_pensjonsvolum DESC
""").fetchdf()

df_naering_scatter


In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    df_naering_scatter["lonsstakere"],
    df_naering_scatter["manedslonn"],
)

plt.title("Lønnstakere og månedslønn per næring")
plt.xlabel("Antall lønnstakere")
plt.ylabel("Månedslønn")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Kort oppsummering

Denne cellen trekker ut noen nøkkelpunkter fra analysen.


In [ ]:
latest_year = df_trend["year"].max()
latest_share = df_trend.loc[
    df_trend["year"] == latest_year,
    "snitt_pensjonsandel_pst",
].iloc[0]

top_kommune = df_kommuner.sort_values(
    "andel_55_pluss",
    ascending=False,
).iloc[0]

top_naering = df_naering.sort_values(
    "estimert_pensjonsvolum_mrd",
    ascending=False,
).iloc[0]

print(f"Siste år i datasettet er {latest_year}.")
print(f"Gjennomsnittlig pensjonsandel er {latest_share:.1f} %.")
print(
    f"Kommunen med høyest andel 55+ er {top_kommune['kommune_label']} "
    f"med {float(top_kommune['andel_55_pluss']):.1f} %."
)
print(
    f"Næringen med høyest estimert pensjonsvolum er "
    f"{top_naering['naering_label']} "
    f"med ca. {top_naering['estimert_pensjonsvolum_mrd']:.1f} mrd. kr."
)


## 9. Valgfritt: lagre grafene som PNG

Denne cellen lagrer de tre viktigste grafene til `/tmp/pensjon_lakehouse/reports/`.


In [ ]:
REPORTS.mkdir(parents=True, exist_ok=True)

# Pensjonsandel over tid
plt.figure(figsize=(10, 5))
plt.plot(df_trend["year"], df_trend["snitt_pensjonsandel_pst"], marker="o")
plt.title("Gjennomsnittlig pensjonsandel over tid")
plt.xlabel("År")
plt.ylabel("Pensjonsandel 55+ (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / "pensjonsandel_trend.png", dpi=150)
plt.close()

# Kommuner med høyest andel 55+
plt.figure(figsize=(10, 6))
plt.barh(df_kommuner["kommune_label"], df_kommuner["andel_55_pluss"])
plt.title("Kommuner med høyest andel innbyggere 55+")
plt.xlabel("Andel 55+ (%)")
plt.ylabel("Kommune")
plt.tight_layout()
plt.savefig(REPORTS / "kommuner_hoyest_andel_55_pluss.png", dpi=150)
plt.close()

# Næringer med høyest estimert pensjonsvolum
plt.figure(figsize=(11, 6))
plt.barh(df_naering["naering_label"], df_naering["estimert_pensjonsvolum_mrd"])
plt.title("Næringer med høyest estimert pensjonsvolum")
plt.xlabel("Estimert pensjonsvolum, mrd. kr")
plt.ylabel("Næring")
plt.tight_layout()
plt.savefig(REPORTS / "naeringer_hoyest_estimert_pensjonsvolum.png", dpi=150)
plt.close()

print(f"Grafer lagret i: {REPORTS}")


## 10. Lukk DuckDB-forbindelsen


In [ ]:
db.close()
